<a href="https://colab.research.google.com/github/nurfnick/Data_Viz/blob/main/Content/Data_Collecting/04_SQL_Essentials.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SQL Essentials

In [ ]:
from google.colab import auth
auth.authenticate_user()
print('Authenticated')

Let's start with the basics.  I'll continue to work with the liquor store data.  

In [ ]:
%%bigquery --project pic-math
SELECT *
FROM `bigquery-public-data.iowa_liquor_sales.sales`
LIMIT 5

Why did I run that command?  Well it gives me an idea of what is in the table to reference and think about what questions I might ask!  Let's see how many gallons of liqour have been sold.

In [ ]:
%%bigquery --project pic-math
SELECT SUM(volume_sold_gallons) as Total_Gallons_of_Liquor
FROM `bigquery-public-data.iowa_liquor_sales.sales`

Let's do a small conversion just to see what that means.  An olympic pool holds 660 000 gallons so

In [ ]:
(80462166.84)/660000

About 121
 swimming pools of liquor in Iowa!  Fun times...

Let's make it more complicated.  Let's see what the dollars per gallon is on the full dataset.

In [ ]:
%%bigquery --project pic-math
SELECT SUM(sale_dollars)/SUM(volume_sold_gallons) as Total_Dollars_Per_Gallon
FROM `bigquery-public-data.iowa_liquor_sales.sales`

Okay not terribly interesting.  Let's take that quesiton and add too it.  Let's create a column that is dollars per gallon of liquor.

In [ ]:
%%bigquery --project pic-math
SELECT sale_dollars/volume_sold_gallons as Dollars_Per_Gallon
FROM `bigquery-public-data.iowa_liquor_sales.sales`
WHERE volume_sold_gallons != 0
LIMIT 5

I have added a few new things here.  The `WHERE` clause allows me to restrict what I consider.  You can combine several of the statements logically

In [ ]:
%%bigquery --project pic-math
SELECT sale_dollars/volume_sold_gallons as Dollars_Per_Gallon,
       item_description,
       sale_dollars as DollarDollaBill
FROM `bigquery-public-data.iowa_liquor_sales.sales`
WHERE volume_sold_gallons != 0 and category_name = 'COFFEE LIQUEURS'

I added the `item_description` so that I could see which ones were different.  It is not utilized the the analysis yet.  Let's include it by getting the average price of coffee liqueurs based on the description.  To do this I'll add the `GROUP BY` command

In [ ]:
%%bigquery --project pic-math
SELECT AVG(sale_dollars/volume_sold_gallons) as Average_Dollars_Per_Gallon, item_description
FROM `bigquery-public-data.iowa_liquor_sales.sales`
WHERE volume_sold_gallons != 0 and category_name = 'COFFEE LIQUEURS'
GROUP BY item_description

Okay but we pointed out earlier that there is no order...  Let's force an order on it

In [ ]:
%%bigquery --project pic-math
SELECT AVG(sale_dollars/volume_sold_gallons) as Average_Dollars_Per_Gallon, item_description
FROM `bigquery-public-data.iowa_liquor_sales.sales`
WHERE volume_sold_gallons != 0 and category_name = 'COFFEE LIQUEURS'
GROUP BY item_description
ORDER BY Average_Dollars_Per_Gallon ASC
LIMIT 1

Let's keep going down the rabbit hole here.  What if we want to rank them?  There are lots of ways `ROW_NUMBER`, `RANK` and `DENSE_RANK`.  I find them difficult to use because they require lots of other inputs.

The general call is something like

``ROW_NUMBER() OVER(PARTITION BY ________ ORDER BY _________)``

Partition is like grouping.  I'll add another liquor to use it

In [ ]:
%%bigquery --project pic-math
SELECT
  AVG(sale_dollars/volume_sold_gallons) as Average_Dollars_Per_Gallon,
  item_description,
  ROW_NUMBER() OVER(
    ORDER BY item_description) row_num
FROM `bigquery-public-data.iowa_liquor_sales.sales`
WHERE volume_sold_gallons != 0 and (category_name = 'COFFEE LIQUEURS')
GROUP BY item_description, category_name
ORDER BY Average_Dollars_Per_Gallon DESC

You should notice that the `ROW_NUMBER` didn't do what we needed.  You will not be able to do the row nor rank on the column we created because it is not yet available to the SQL call.  This leads to sub-processees.  Let's show one today and come back to it next class.

In [ ]:
%%bigquery --project pic-math
WITH t as(
SELECT
  AVG(sale_dollars/volume_sold_gallons) as Average_Dollars_Per_Gallon,
  item_description, category_name
FROM `bigquery-public-data.iowa_liquor_sales.sales`
WHERE volume_sold_gallons != 0 and (category_name = 'COFFEE LIQUEURS' or category_name = 'IMPORTED VODKAS')
GROUP BY item_description, category_name
ORDER BY Average_Dollars_Per_Gallon DESC
)

SELECT *,
    RANK() OVER(
    PARTITION BY category_name
    ORDER BY Average_Dollars_Per_Gallon) rk_num
FROM t
ORDER BY rk_num

We will come back to this but a nice taste of some of the really powerful aspects of SQL!

## Dealing With Strings

I wanted to find why 'vodka' wasn't in the table.  Well it was but with other qualifiers.  I needed to use a string command `LIKE` I also looked at `CONTAINS` but didn't get it to work.

In [ ]:
%%bigquery --project pic-math
SELECT category_name
FROM `bigquery-public-data.iowa_liquor_sales.sales`
WHERE category_name LIKE '%VODKA%'
GROUP BY category_name


## Your Turn



Assignement for today

1. Start a notebook getting BigQuery to work. Feel free to use the authentication atop.
2. Navigate to the dataset 'austin_bikeshare.bikeshare_trips'
3. Compute the average time for a trip based on starting point
4. Compute how many trips start at each starting point.
5.


